# Cheat Sheet

This is a collection of quickfire tutorials to help you get started with `PortfolioOptimisers.jl` without delving into the examples and/or documentation.

## 1. Downloading data

There are both open and closed source financial data APIs, in Julia we have [`YFinance.jl`](https://github.com/eohne/YFinance.jl) and [`MarketData.jl`](https://github.com/JuliaQuant/MarketData.jl), both of which are quite good.

## 2. Computing returns

Usually, price data is obtained using an API, and the returns have to be computed. In `PortfolioOptimisers.jl`, we have `prices_to_returns`, which handles asset, factor, and benchmark price data, as well as implied volatilities, and volatility premiums. It performs appropriate data validation checks to ensure the timestamps match, and the data is clean. It can also preprocess missing data, fill gaps using [`Impute.jl`](https://github.com/invenia/Impute.jl), as well as collapse the data to lower frequencies using [`TimeSeries.jl`](https://github.com/JuliaStats/TimeSeries.jl).

Here we show a quick example of a heterogenous dataset which will only return the data with matching timestamps.

In [1]:
using PortfolioOptimisers, CSV, TimeSeries, DataFrames, PrettyTables, LinearAlgebra,
      StableRNGs
resfmt = (v, i, j) -> begin
    return if j == 1
        v
    else
        isa(v, AbstractFloat) ? "$(round(v*100, digits=3)) %" : v
    end
end;

rd = prices_to_returns(TimeArray(CSV.File(joinpath(@__DIR__, "SP500.csv.gz"));
                                 timestamp = :Date)[(end - 7 * 252):(end - 252 * 2)],
                       TimeArray(CSV.File(joinpath(@__DIR__, "Factors.csv.gz"));
                                 timestamp = :Date)[(end - 6 * 252):(end - 252)];
                       B = TimeArray(CSV.File(joinpath(@__DIR__, "SP500_idx.csv.gz"));
                                     timestamp = :Date)[(end - 5 * 252):end])

ReturnsResult
    nx ┼ 20-element Vector{String}
     X ┼ 756×20 Matrix{Float64}
    nf ┼ Vector{String}: ["MTUM", "QUAL", "SIZE", "USMV", "VLUE"]
     F ┼ 756×5 Matrix{Float64}
    nb ┼ Vector{String}: ["SP500"]
     B ┼ 756-element Vector{Float64}
    ts ┼ 756-element Vector{Date}
    iv ┼ nothing
  ivpa ┴ nothing


## 3. Basic Optimisations

There are many optimisers available in `PortfolioOptimisers.jl`. Here we will showcase their basic usage.

### 3.1 Naive Optimisers

Naive optimisers do not use sophisticated optimisation algorithms, but rather very basic ones that offer robustness and diversificaiton by virtue of being unsophisticated.

#### 3.1.1 Inverse volatility

This uses the diagonal of the covariance to set the weights, if the flag `sq` is true, the weights are just the inverse of each entry in the diagonal, else it is the inverse of the square root of each entry in the diagonal.

In [2]:
variance = diag(prior(EmpiricalPrior(), rd).sigma)

res1 = optimise(InverseVolatility(), rd)
res2 = optimise(InverseVolatility(; sq = true), rd)
inv_vol = 1 ./ sqrt.(variance)
inv_vol /= sum(inv_vol)
inv_var = 1 ./ variance
inv_var /= sum(inv_var)
pretty_table(DataFrame([rd.nx res1.w inv_vol res2.w inv_var],
                       ["assets", "Opt Vol", "Inv Vol", "Opt Var", "Inv Var"]);
             formatters = [resfmt])

┌────────┬─────────┬─────────┬─────────┬─────────┐
│ assets │ Opt Vol │ Inv Vol │ Opt Var │ Inv Var │
│    Any │     Any │     Any │     Any │     Any │
├────────┼─────────┼─────────┼─────────┼─────────┤
│   AAPL │ 4.555 % │ 4.555 % │  3.86 % │  3.86 % │
│    AMD │  2.69 % │  2.69 % │ 1.346 % │ 1.346 % │
│    BAC │ 4.088 % │ 4.088 % │ 3.108 % │ 3.108 % │
│    BBY │ 3.955 % │ 3.955 % │ 2.909 % │ 2.909 % │
│    CVX │ 4.059 % │ 4.059 % │ 3.064 % │ 3.064 % │
│     GE │  3.25 % │  3.25 % │ 1.965 % │ 1.965 % │
│     HD │  5.29 % │  5.29 % │ 5.205 % │ 5.205 % │
│    JNJ │ 6.787 % │ 6.787 % │ 8.569 % │ 8.569 % │
│    JPM │ 4.492 % │ 4.492 % │ 3.753 % │ 3.753 % │
│     KO │ 6.689 % │ 6.689 % │ 8.322 % │ 8.322 % │
│    LLY │ 5.282 % │ 5.282 % │ 5.189 % │ 5.189 % │
│    MRK │ 6.621 % │ 6.621 % │ 8.154 % │ 8.154 % │
│   MSFT │  4.96 % │  4.96 % │ 4.576 % │ 4.576 % │
│    PEP │ 6.402 % │ 6.402 % │ 7.625 % │ 7.625 % │
│    PFE │ 6.147 % │ 6.147 % │ 7.029 % │ 7.029 % │
│      ⋮ │       ⋮ │       ⋮ │ 

#### 3.1.2 Equal weighted

This assigns equal weights to all assets.

In [3]:
res = optimise(EqualWeighted(), rd)
pretty_table(DataFrame([rd.nx res.w], ["assets", "Weights"]); formatters = [resfmt])

┌────────┬─────────┐
│ assets │ Weights │
│    Any │     Any │
├────────┼─────────┤
│   AAPL │   5.0 % │
│    AMD │   5.0 % │
│    BAC │   5.0 % │
│    BBY │   5.0 % │
│    CVX │   5.0 % │
│     GE │   5.0 % │
│     HD │   5.0 % │
│    JNJ │   5.0 % │
│    JPM │   5.0 % │
│     KO │   5.0 % │
│    LLY │   5.0 % │
│    MRK │   5.0 % │
│   MSFT │   5.0 % │
│    PEP │   5.0 % │
│    PFE │   5.0 % │
│      ⋮ │       ⋮ │
└────────┴─────────┘
      5 rows omitted


#### 3.1.3 Random weighted

This randomly assigns weights according to a [`Dirichlet`](https://juliastats.org/Distributions.jl/latest/multivariate/#Distributions.Dirichlet) distribution. It's possible to provide a custom alpha parameter as a vector or number, random number generator, and seed.

In [4]:
res = optimise(RandomWeighted(; alpha = 1, rng = StableRNG(696), seed = 66420), rd)
pretty_table(DataFrame([rd.nx res.w], ["assets", "Weights"]); formatters = [resfmt])

┌────────┬──────────┐
│ assets │  Weights │
│    Any │      Any │
├────────┼──────────┤
│   AAPL │  2.736 % │
│    AMD │ 10.553 % │
│    BAC │    3.0 % │
│    BBY │   3.64 % │
│    CVX │  7.897 % │
│     GE │  7.176 % │
│     HD │  0.461 % │
│    JNJ │  0.014 % │
│    JPM │  3.476 % │
│     KO │  9.838 % │
│    LLY │   1.41 % │
│    MRK │  1.049 % │
│   MSFT │  9.721 % │
│    PEP │ 17.372 % │
│    PFE │   2.36 % │
│      ⋮ │        ⋮ │
└────────┴──────────┘
       5 rows omitted


### 3.2 JuMP optimisers

The JuMP-based optimisers use traditional mathematical optimisation. As such, they are the most flexible when it comes to constraints, and for those which accept them, objective functions. Most risk measures are also compatible with these, aside from a few exclusively compatible with clustering optimisations, as well as other risk measures which are incompatible with any optimisation. All JuMP-based optimisers require the user to provide a JuMP-compatible solver, which supports the type of constraints being used.

If using open-source solvers we recommend [`Clarabel`](https://github.com/oxfordcontrol/Clarabel.jl) when not using MIP constraints. When using MIP constraints, [`Pajarito`](https://github.com/jump-dev/Pajarito.jl) with [`Clarabel`](https://github.com/oxfordcontrol/Clarabel.jl) as the continuous solver, and [`HiGHS`](https://github.com/jump-dev/HiGHS.jl) as the MIP solver.

Users can provide a vector of solvers which will be iterated over until one solves the problem satisfactorily, or all fail. Other than optimisation-specific constraints, general constraints are applied at the level of the [`JuMPOptimiser`]-(@ref), whether the problem is feasable or not depends on the specific constraints and the provided solver's support for constraint types/ability to solve the problem.

In [5]:
using Clarabel
slv = [Solver(; name = :clarabel1, solver = Clarabel.Optimizer,
              check_sol = (; allow_local = true, allow_almost = true),
              settings = "verbose" => false),
       Solver(; name = :clarabel3, solver = Clarabel.Optimizer,
              check_sol = (; allow_local = true, allow_almost = true),
              settings = Dict("verbose" => false, "max_step_fraction" => 0.9)),
       Solver(; name = :clarabel5, solver = Clarabel.Optimizer,
              check_sol = (; allow_local = true, allow_almost = true),
              settings = Dict("verbose" => false, "max_step_fraction" => 0.80)),
       Solver(; name = :clarabel7, solver = Clarabel.Optimizer,
              check_sol = (; allow_local = true, allow_almost = true),
              settings = Dict("verbose" => false, "max_step_fraction" => 0.7)),
       Solver(; name = :clarabel8, solver = Clarabel.Optimizer,
              check_sol = (; allow_local = true, allow_almost = true),
              settings = Dict("verbose" => false, "max_step_fraction" => 0.6,
                              "max_iter" => 1500, "tol_gap_abs" => 1e-4,
                              "tol_gap_rel" => 1e-4, "tol_ktratio" => 1e-3,
                              "tol_feas" => 1e-4, "tol_infeas_abs" => 1e-4,
                              "tol_infeas_rel" => 1e-4, "reduced_tol_gap_abs" => 1e-4,
                              "reduced_tol_gap_rel" => 1e-4, "reduced_tol_ktratio" => 1e-3,
                              "reduced_tol_feas" => 1e-4, "reduced_tol_infeas_abs" => 1e-4,
                              "reduced_tol_infeas_rel" => 1e-4))];

#### 3.2.1 Mean Risk

[`MeanRisk`]-(@ref) is the traditional portfolio optimisation problem. It seeks to minimise the risk with respect to a target return, or maximise the return with respect to a target risk. It supports four objective functions via the `obj` keyword which defaults to [`MinimumRisk`]-(@ref) which use the relationship between risk and returns in different ways, and the risk measure(s) are specified with the `r` keyword which defaults to `Variance`.

In [6]:
# Minimum risk (default)
mr1 = MeanRisk(; obj = MinimumRisk(), opt = JuMPOptimiser(; slv = slv))
# Maximum utility
mr2 = MeanRisk(; obj = MaximumUtility(), opt = JuMPOptimiser(; slv = slv))
# Maximum risk adjusted return ratio
mr3 = MeanRisk(; obj = MaximumRatio(), opt = JuMPOptimiser(; slv = slv))
# Maximum return
mr4 = MeanRisk(; obj = MaximumReturn(), opt = JuMPOptimiser(; slv = slv))
# Optimise all objective functions at once
ress = optimise.([mr1, mr2, mr3, mr4], rd);

`PortfolioOptimisers.jl` provides users with the ability to use multiple risk measures per optimisation, which means that some risk measure have to keep certain internal statistics to be able to compute the risk. We provide factory functions that create the risk measures with the appropraite internal statistics.

The [`MeanRisk`]-(@ref) optimiser defaults to the variance so we will use that to compute the risk statistics. All optimisations use the same prior estimator, as well as portfolio return estimator so we will use only the first

In [7]:
pr = ress[1].pr
r = factory(Variance(), pr)
ret = mr1.opt.ret
rk_rt_ratio = [expected_risk_ret_ratio(r, ret, res.w, pr) for res in ress]
rk = map(rr -> rr[1], rk_rt_ratio)
rt = map(rr -> rr[2], rk_rt_ratio)
ratio = map(rr -> rr[3], rk_rt_ratio)

# Display results
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [res.w for res in ress]),
                            [:MinimumRisk, :MaximumUtility, :MaximumRatio, :MaximumReturn]));
             formatters = [resfmt])
pretty_table(hcat(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"]),
                  DataFrame(vcat(rk', rt', ratio'),
                            [:MinimumRisk, :MaximumUtility, :MaximumRatio, :MaximumReturn]));
             formatters = [resfmt])

┌────────┬─────────────┬────────────────┬──────────────┬───────────────┐
│ assets │ MinimumRisk │ MaximumUtility │ MaximumRatio │ MaximumReturn │
│ String │     Float64 │        Float64 │      Float64 │       Float64 │
├────────┼─────────────┼────────────────┼──────────────┼───────────────┤
│   AAPL │       0.0 % │       32.448 % │     31.784 % │         0.0 % │
│    AMD │       0.0 % │        46.15 % │     37.264 % │       100.0 % │
│    BAC │       0.0 % │          0.0 % │        0.0 % │         0.0 % │
│    BBY │     0.001 % │          0.0 % │        0.0 % │         0.0 % │
│    CVX │       0.0 % │          0.0 % │        0.0 % │         0.0 % │
│     GE │       0.0 % │          0.0 % │        0.0 % │         0.0 % │
│     HD │       0.0 % │          0.0 % │        0.0 % │         0.0 % │
│    JNJ │    13.325 % │          0.0 % │        0.0 % │         0.0 % │
│    JPM │       0.0 % │          0.0 % │        0.0 % │         0.0 % │
│     KO │    20.013 % │          0.0 % │        0.

#### 3.2.2 Factor Risk Contribution

This is a more advanced estimator, it requires some more set up. It allows users to provide objective functions, but also define risk contributions per factor to the variance risk measure. The minimum risk optimisaion will follow the risk contribution constraints the closest, and with enough data and assets can be quite exact up to the user provided convergence settings for the provided solvers.

It is compatible with other risk measures, but only the variance can take risk contribution constraints, without them or when using other risk measures it is largely the same as the [`MeanRisk`]-(@ref) estimator.

First we need to provide a instance of `AssetSets` which defines sets of assets or factors and their relationships, which lets `PortfolioOptimisers.jl` create linear constraints according to how users define them via `LinearConstraintEstimator`. This way it's possible to define groups of assets/factors and how they relate to each other. We will showcase them later. For now, we need these to define the relationship between factors for their risk contribution.

The `AssetSets` has a `key` property which defines the default search key in `dict`, `dict` must contain a key matching `key` whose value is taken to tbe the names of the assets/factors around which the sets are defined.

In this case we use this to define the set of factors with key "nf" (default "nx"). We use these to define the factor risk contribution constraints, and we have to assign the linear constraint estimator for the risk contribution to the `Variance`. We'll define the constraints such that each factor contributes between 30% and 12% of the total variance risk.

In [8]:
sets = AssetSets(; key = "nf", dict = Dict("nf" => rd.nf))
lcs = LinearConstraintEstimator(;
                                val = [["$f <= 0.3" for f in rd.nf];
                                       ["$f >= 0.12" for f in rd.nf]])
r = Variance(; rc = lcs)

Variance
  settings ┼ RiskMeasureSettings
           │   scale ┼ Float64: 1.0
           │      ub ┼ nothing
           │     rke ┴ Bool: true
     sigma ┼ nothing
      chol ┼ nothing
        rc ┼ LinearConstraintEstimator
           │   val ┼ 10-element Vector{String}
           │   key ┴ nothing
       alg ┴ SquaredSOCRiskExpr()


We can optimise for the different objective functions.

In [9]:
# Minimum risk (default)
frc1 = FactorRiskContribution(; r = r, obj = MinimumRisk(), sets = sets,
                              opt = JuMPOptimiser(; slv = slv))
# Maximum utility, `l` controls the risk aversion.
frc2 = FactorRiskContribution(; r = r, obj = MaximumUtility(; l = 8), sets = sets,
                              opt = JuMPOptimiser(; slv = slv))
# Maximum risk adjusted return ratio, rf is the risk free rate.
frc3 = FactorRiskContribution(; r = r, obj = MaximumRatio(; rf = 4.2 / 252 / 100),
                              sets = sets, opt = JuMPOptimiser(; slv = slv))
# Maximum return
frc4 = FactorRiskContribution(; r = r, obj = MaximumReturn(), sets = sets,
                              opt = JuMPOptimiser(; slv = slv))
# Optimise all objective functions at once
ress = optimise.([frc1, frc2, frc3, frc4], rd);

We can display the results and factor risk contributions [`factor_risk_contribution`]-(@ref), but we have to normalise them using their sum. Again we use the factory function to set the appropriate internal parameters. The last entry in the risk contribution is the regression intercept.

In [10]:
r = factory(r, pr)
rkcs = [factor_risk_contribution(r, res.w, pr.X; rd = rd) for res in ress]
rkcs = rkcs ./ sum.(rkcs)

pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [res.w for res in ress]),
                            [:MinimumRisk, :MaximumUtility, :MaximumRatio, :MaximumReturn]));
             formatters = [resfmt])
pretty_table(hcat(DataFrame(:factors => [rd.nf; "Intercept"]),
                  DataFrame(reduce(hcat, rkcs),
                            ["RC MinRisk", "RC Max Util", "RC Max Ratio", "RC Max Ret"]));
             formatters = [resfmt])

┌────────┬─────────────┬────────────────┬──────────────┬───────────────┐
│ assets │ MinimumRisk │ MaximumUtility │ MaximumRatio │ MaximumReturn │
│ String │     Float64 │        Float64 │      Float64 │       Float64 │
├────────┼─────────────┼────────────────┼──────────────┼───────────────┤
│   AAPL │      1.12 % │        1.382 % │      8.372 % │      11.873 % │
│    AMD │     8.315 % │        12.22 % │     12.164 % │      21.114 % │
│    BAC │     2.779 % │        0.618 % │      4.591 % │       1.162 % │
│    BBY │       4.1 % │        3.574 % │      5.802 % │       5.427 % │
│    CVX │     9.866 % │       10.406 % │        0.0 % │       5.296 % │
│     GE │     4.071 % │          2.3 % │      1.443 % │         0.0 % │
│     HD │    10.868 % │       13.242 % │      3.257 % │      10.359 % │
│    JNJ │     1.199 % │          0.0 % │      5.253 % │       1.335 % │
│    JPM │     4.265 % │        2.667 % │       2.85 % │       1.196 % │
│     KO │     6.997 % │        6.761 % │      3.08

#### 3.2.3 Near Optimal Centering

[`NearOptimalCentering`]-(@ref) is a way to smear an optimal portfolio accross a region around the point of optimality. The size of this region can be tuend by the user via the `bins` keyword, or automatically decided based on the number of observations and assets (default). This makes the portfolio more robust to estimation error and more diversified. It is not compatible with risk measures which produce quadratic risk expressions, so the risk measure keyword `r` defaults to `StandardDeviation`.

There are two variants, defined by the `alg` keyword, one which applies all constraints to the inner [`MeanRisk`]-(@ref) optimisation and leave the near optimal portfolio to fall where it may [`UnconstrainedNearOptimalCentering`]-(@ref), meaning the constraints will not be satisfied by the near optimal portfolio. And another which does apply the constraints to the near optimal portfolio [`ConstrainedNearOptimalCentering`]-(@ref).

In [11]:
# Minimum risk (default)
noc1 = NearOptimalCentering(; obj = MinimumRisk(), opt = JuMPOptimiser(; slv = slv))
# Maximum utility
noc2 = NearOptimalCentering(; obj = MaximumUtility(; l = 0.5),
                            opt = JuMPOptimiser(; slv = slv))
# Maximum risk adjusted return ratio
noc3 = NearOptimalCentering(; obj = MaximumRatio(), opt = JuMPOptimiser(; slv = slv))
# Maximum return
noc4 = NearOptimalCentering(; obj = MaximumReturn(), opt = JuMPOptimiser(; slv = slv))
# Optimise all objective functions at once
ress = optimise.([noc1, noc2, noc3, noc4], rd);

View and compute the results.

In [12]:
pr = ress[1].pr
r = factory(StandardDeviation(), pr)
ret = mr1.opt.ret
rk_rt_ratio = [expected_risk_ret_ratio(r, ret, res.w, pr) for res in ress]
rk = map(rr -> rr[1], rk_rt_ratio)
rt = map(rr -> rr[2], rk_rt_ratio)
ratio = map(rr -> rr[3], rk_rt_ratio)
# Display results
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [res.w for res in ress]),
                            [:MinimumRisk, :MaximumUtility, :MaximumRatio, :MaximumReturn]));
             formatters = [resfmt])
pretty_table(hcat(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"]),
                  DataFrame(vcat(rk', rt', ratio'),
                            [:MinimumRisk, :MaximumUtility, :MaximumRatio, :MaximumReturn]));
             formatters = [resfmt])

┌────────┬─────────────┬────────────────┬──────────────┬───────────────┐
│ assets │ MinimumRisk │ MaximumUtility │ MaximumRatio │ MaximumReturn │
│ String │     Float64 │        Float64 │      Float64 │       Float64 │
├────────┼─────────────┼────────────────┼──────────────┼───────────────┤
│   AAPL │      2.27 % │        3.728 % │     18.433 % │       0.241 % │
│    AMD │     1.479 % │        3.318 % │     42.803 % │      97.341 % │
│    BAC │      1.28 % │        1.393 % │       0.88 % │       0.125 % │
│    BBY │     2.149 % │         2.45 % │      1.479 % │       0.154 % │
│    CVX │     1.479 % │         1.43 % │      0.671 % │       0.109 % │
│     GE │     1.653 % │        1.546 % │      0.675 % │       0.108 % │
│     HD │     2.472 % │        2.628 % │      1.303 % │       0.141 % │
│    JNJ │     8.507 % │        6.183 % │      1.376 % │       0.123 % │
│    JPM │       1.7 % │        1.905 % │      1.199 % │       0.133 % │
│     KO │    10.764 % │        8.822 % │      1.81

#### 3.2.4 Risk Budgeting

[`RiskBudgeting`]-(@ref) provides a way to allocate risk accross assets or factors via the `rba` keyword according to a user-defined risk budgeting vector provided via the `rkb` keyword of [`AssetRiskBudgeting`]-(@ref) and [`FactorRiskBudgeting`]-(@ref) risk budgeting algorithms. The risk budget vectors do not have to be normalised. The risk being budgeted depends on the risk measures used. This does not support objective functions, the optimisation is solely focused on achieving the risk budgeting as closely as possible. It is compatible with the same risk measures as [`MeanRisk`]-(@ref).

##### 3.2.4.1 Asset Risk Budgeting

This version allocated risk accross assets.

In [13]:
r = Variance()
# Equal risk contribution per asset (default)
rba1 = RiskBudgeting(; r = r,
                     rba = AssetRiskBudgeting(;
                                              rkb = RiskBudget(;
                                                               val = fill(1.0,
                                                                          length(rd.nx)))),
                     opt = JuMPOptimiser(; slv = slv))
# Increasing risk contribution per asset
rba2 = RiskBudgeting(; r = r,
                     rba = AssetRiskBudgeting(; rkb = RiskBudget(; val = 1:length(rd.nx))),
                     opt = JuMPOptimiser(; slv = slv))
# Optimise all risk budgeting estimators at once
ress = optimise.([rba1, rba2], rd);

View and compute the results.

In [14]:
r = factory(r, pr)
rkcs = [risk_contribution(r, res.w, pr.X) for res in ress]
rkcs = rkcs ./ sum.(rkcs)
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [[res.w rkc] for (res, rkc) in zip(ress, rkcs)]),
                            ["Eq Risk Weights", "Eq Risk Budget", "Incr Risk Weights",
                             "Incr Risk Budget"])); formatters = [resfmt])

┌────────┬─────────────────┬────────────────┬───────────────────┬───────────────
│ assets │ Eq Risk Weights │ Eq Risk Budget │ Incr Risk Weights │ Incr Risk Bu ⋯
│ String │         Float64 │        Float64 │           Float64 │          Flo ⋯
├────────┼─────────────────┼────────────────┼───────────────────┼───────────────
│   AAPL │         4.386 % │          5.0 % │           0.429 % │          0.4 ⋯
│    AMD │         3.475 % │          5.0 % │           0.732 % │          0.9 ⋯
│    BAC │         3.611 % │          5.0 % │           1.027 % │          1.4 ⋯
│    BBY │         4.226 % │          5.0 % │           1.675 % │          1.9 ⋯
│    CVX │         3.773 % │          5.0 % │           1.765 % │          2.3 ⋯
│     GE │         3.806 % │          5.0 % │            2.22 % │          2.8 ⋯
│     HD │         4.705 % │          5.0 % │           3.129 % │          3.3 ⋯
│    JNJ │         6.437 % │          5.0 % │           4.603 % │          3.8 ⋯
│    JPM │         3.982 % │

##### 3.2.4.1 Factor Risk Budgeting

This version allocated risk accross factors.

In [15]:
r = Variance()
# Equal risk contribution per factor (default)
rba1 = RiskBudgeting(; r = r,
                     rba = FactorRiskBudgeting(;
                                               rkb = RiskBudget(;
                                                                val = range(; start = 1,
                                                                            stop = 1,
                                                                            length = length(rd.nf)))),
                     opt = JuMPOptimiser(; slv = slv))
# Increasing risk contribution per factor
rba2 = RiskBudgeting(; r = r,
                     rba = FactorRiskBudgeting(; rkb = RiskBudget(; val = 1:length(rd.nf))),
                     opt = JuMPOptimiser(; slv = slv))
# Optimise all risk budgeting estimators at once
ress = optimise.([rba1, rba2], rd);

View and compute the results.

In [16]:
r = factory(r, pr)
rkcas = [risk_contribution(r, res.w, pr.X) for res in ress]
rkcas = rkcas ./ sum.(rkcas)
rkcfs = [factor_risk_contribution(r, res.w, pr.X; rd = rd) for res in ress]
rkcfs = rkcfs ./ sum.(rkcfs)
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [[res.w rkc] for (res, rkc) in zip(ress, rkcas)]),
                            ["Eq Risk Weights", "Eq Risk Budget", "Incr Risk Weights",
                             "Incr Risk Budget"])); formatters = [resfmt],
             title = "Asset risk contribution")
pretty_table(hcat(DataFrame(:factors => [rd.nf; "Intercept"]),
                  DataFrame(reduce(hcat,
                                   [[[(res.prb.b1 \ res.w); NaN] rkc]
                                    for (res, rkc) in zip(ress, rkcfs)]),
                            ["Eq Risk Weights", "Eq Risk Budget", "Incr Risk Weights",
                             "Incr Risk Budget"])); formatters = [resfmt],
             title = "Factor risk contribution")

                            Asset risk contribution
┌────────┬─────────────────┬────────────────┬───────────────────┬───────────────
│ assets │ Eq Risk Weights │ Eq Risk Budget │ Incr Risk Weights │ Incr Risk Bu ⋯
│ String │         Float64 │        Float64 │           Float64 │          Flo ⋯
├────────┼─────────────────┼────────────────┼───────────────────┼───────────────
│   AAPL │         2.518 % │        2.617 % │             0.0 % │            0 ⋯
│    AMD │         3.202 % │        4.295 % │           3.644 % │          4.7 ⋯
│    BAC │         2.928 % │        3.596 % │           9.572 % │         12.5 ⋯
│    BBY │         1.884 % │        2.013 % │           3.022 % │          3.2 ⋯
│    CVX │         8.983 % │       11.033 % │           7.751 % │          9.7 ⋯
│     GE │         2.945 % │        3.334 % │           4.587 % │          5.6 ⋯
│     HD │        22.721 % │       24.027 % │           19.13 % │         19.7 ⋯
│    JNJ │           0.0 % │          0.0 % │            

#### 3.2.5 Relaxed Risk Budgeting

[`RelaxedRiskBudgeting`]-(@ref) provides a way to allocate risk accross assets or factors according to a user-defined risk budgeting vector, which does not have to be normalised. They are provided in the same way as for [`RiskBudgeting`]-(@ref), it does not accept risk measures as it's only available for the variance, and it will not follow the risk budget as closely.

There are three variants, basic, regularised, and regularised and penalised. Since the asset and factor versions are the same as before we will only show the different relaxed risk budgeting algorithms.

In [17]:
# Basic
rrb1 = RelaxedRiskBudgeting(;
                            rba = AssetRiskBudgeting(;
                                                     rkb = RiskBudget(;
                                                                      val = range(;
                                                                                  start = 1,
                                                                                  stop = 1,
                                                                                  length = length(rd.nx)))),
                            alg = BasicRelaxedRiskBudgeting(),
                            opt = JuMPOptimiser(; slv = slv))
# Regularised
rrb2 = RelaxedRiskBudgeting(;
                            rba = AssetRiskBudgeting(;
                                                     rkb = RiskBudget(;
                                                                      val = range(;
                                                                                  start = 1,
                                                                                  stop = 1,
                                                                                  length = length(rd.nx)))),
                            alg = RegularisedRelaxedRiskBudgeting(),
                            opt = JuMPOptimiser(; slv = slv))
# Regularised and penalised, `p` is the penalty factor (default 1)
rrrb3 = RelaxedRiskBudgeting(;
                             rba = AssetRiskBudgeting(;
                                                      rkb = RiskBudget(;
                                                                       val = range(;
                                                                                   start = 1,
                                                                                   stop = 1,
                                                                                   length = length(rd.nx)))),
                             alg = RegularisedPenalisedRelaxedRiskBudgeting(; p = 1),
                             opt = JuMPOptimiser(; slv = slv))
ress = optimise.([rrb1, rrb2, rrrb3], rd);

View and compute the results.

In [18]:
r = Variance()
r = factory(r, pr)
rkcs = [risk_contribution(r, res.w, pr.X) for res in ress]
rkcs = rkcs ./ sum.(rkcs)
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [[res.w rkc] for (res, rkc) in zip(ress, rkcs)]),
                            ["B Weights", "B Budget", "Reg Weights", "Reg Budget",
                             "RegPen Weights", "RegPen Budget"])); formatters = [resfmt])

┌────────┬───────────┬──────────┬─────────────┬────────────┬────────────────┬───
│ assets │ B Weights │ B Budget │ Reg Weights │ Reg Budget │ RegPen Weights │  ⋯
│ String │   Float64 │  Float64 │     Float64 │    Float64 │        Float64 │  ⋯
├────────┼───────────┼──────────┼─────────────┼────────────┼────────────────┼───
│   AAPL │   2.445 % │   2.92 % │     1.697 % │    2.002 % │        2.969 % │  ⋯
│    AMD │   2.028 % │   2.92 % │     1.445 % │    2.002 % │        2.431 % │  ⋯
│    BAC │   2.088 % │   2.92 % │     1.463 % │    2.002 % │        2.494 % │  ⋯
│    BBY │   2.434 % │   2.92 % │     1.718 % │    2.002 % │        2.926 % │  ⋯
│    CVX │   2.214 % │   2.92 % │     1.558 % │    2.002 % │         2.63 % │  ⋯
│     GE │   2.282 % │   2.92 % │     1.622 % │    2.002 % │        2.683 % │  ⋯
│     HD │   2.583 % │   2.92 % │      1.78 % │    2.002 % │        3.157 % │  ⋯
│    JNJ │   7.688 % │  6.895 % │     9.706 % │    9.151 % │        9.168 % │  ⋯
│    JPM │   2.292 % │   2.9

### 3.3 Clustering optimisers

Clustering based optimisers use the relatedness of the assets and the risks associated with these structures to assign weights based on those risks. Aside from the [`NestedClusters`]-(@ref) estimator, they all use a [`HierarchicalOptimiser`]-(@ref) in much the same way that JuMP-based optimisers use [`JuMPOptimiser`]-(@ref). In this case, the [`HierarchicalOptimiser`]-(@ref) does not require a solver to be specified via the `slv` keyword unless it uses a risk measure that requires one.

#### 3.3.1 Hierachical risk parity

The [`HierarchicalRiskParity`]-(@ref) estimator uses the hierarchical structure of the assets to iteratively partition the assets into smaller and smaller clusters computing the weights as a function of the risk between left and right cluster at each partition level. This accepts the same risk measures as JuMP based optimisers as well as some extra ones for which there are no traditional optimisation formulations.

The [`HierarchicalOptimiser`]-(@ref) is specified via the `opt` keyword and the risk measure can be specified via the `r` keyword which defaults to the `Variance`.

In [19]:
# Hierarchical risk parity
hrp = HierarchicalRiskParity()
res = optimise(hrp, rd)

HierarchicalResult
       oe ┼ DataType: DataType
       pr ┼ LowOrderPrior
          │         X ┼ 756×20 Matrix{Float64}
          │        mu ┼ 20-element Vector{Float64}
          │     sigma ┼ 20×20 Matrix{Float64}
          │      chol ┼ nothing
          │         w ┼ nothing
          │       ens ┼ nothing
          │       kld ┼ nothing
          │        ow ┼ nothing
          │        rr ┼ nothing
          │      f_mu ┼ nothing
          │   f_sigma ┼ nothing
          │       f_w ┴ nothing
      clr ┼ Clusters
          │   res ┼ Clustering.Hclust{Float64}([-13 -1; -16 -14; -7 -4; -5 -20; -12 -8; -3 -9; -15 5; 7 -11; -18 1; 9 3; 2 -10; 10 -2; -19 11; 6 4; -6 14; 15 -17; 13 8; 17 12; 18 16], [0.3396697691642121, 0.34762217459549366, 0.4294860942569611, 0.2816354347902738, 0.4214694877772248, 0.17325019149889836, 0.4275113407151469, 0.45669085435759316, 0.4918776056007628, 0.5138138228514841, 0.4134598069341204, 0.5586690778496974, 0.5383772425351973, 0.47369740075876027, 0.

View and compute the results.

In [20]:
pr = res.pr
r = factory(Variance(), pr)
rk, rt, ratio = expected_risk_ret_ratio(r, ArithmeticReturn(), res.w, pr)
# Display results
pretty_table(DataFrame(:assets => rd.nx, :Weights => res.w); formatters = [resfmt])
pretty_table(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"],
                       :Measure => [rk, rt, ratio]); formatters = [resfmt])

┌────────┬─────────┐
│ assets │ Weights │
│ String │ Float64 │
├────────┼─────────┤
│   AAPL │ 4.612 % │
│    AMD │ 1.378 % │
│    BAC │ 3.001 % │
│    BBY │ 3.707 % │
│    CVX │ 3.186 % │
│     GE │ 2.013 % │
│     HD │ 6.219 % │
│    JNJ │ 9.316 % │
│    JPM │ 3.624 % │
│     KO │ 5.824 % │
│    LLY │ 5.857 % │
│    MRK │ 8.865 % │
│   MSFT │ 3.198 % │
│    PEP │ 8.275 % │
│    PFE │ 4.918 % │
│      ⋮ │       ⋮ │
└────────┴─────────┘
      5 rows omitted
┌─────────────────┬───────────┐
│            Stat │   Measure │
│          String │   Float64 │
├─────────────────┼───────────┤
│        Variance │   0.018 % │
│          Return │    0.07 % │
│ Return/Variance │ 392.695 % │
└─────────────────┴───────────┘


#### 3.3.2 Schur complement hierarchical risk parity

The [`SchurComplementHierarchicalRiskParity`]-(@ref) estimator works similarly to [`HierarchicalRiskParity`]-(@ref), but uses the Schur complement to compute decide whether to include more information into the risk calculation. It is only available for the variance and standard deviation because it relies on the Schur complement. It serves almost as an interpolation between the classic mean variance optimisation and hierarchical risk parity. It has a `params` keyword which allows users to specify an instance or vector of instances of [`SchurComplementParams`]-(@ref) which specify the risk measure, the "interpolation" parameter `gamma ∈ [0, 1]`, and whether the algorithm should be kept monotonic in risk as `gamma` increases (default).

When `gamma` is zero, it reduces to the [`HierarchicalRiskParity`]-(@ref) estimator, the closer it is to one the closer it is to the classic mean variance optimisation. There is no need to specify all these parameters, as they all have default values, this is for demonstration purposes. It's worth noting that there are values of `gamma` for which the Schur augmented matrix is not positive definite as it cannot add any more risk information beyond a certain point, so when wanting `gamma` to be large, one should use [`NonMonotonicSchurComplement`]-(@ref) and make sure to disable the positive definite projection in the `pdm` keyword of [`SchurComplementParams`]-(@ref.

In [21]:
r = Variance()
# Hierachical risk parity
hrp = HierarchicalRiskParity()
# Schur complement hierarchical risk parity converging to the hierarchical risk parity
sch1 = SchurComplementHierarchicalRiskParity(;
                                             params = SchurComplementParams(; gamma = 0,
                                                                            r = r,
                                                                            alg = MonotonicSchurComplement()))
# Mean variance optimisation
mr = MeanRisk(; opt = JuMPOptimiser(; slv = slv))
# Schur complement hierarchical risk parity nearing the mean variance optimisation, no positive definite projection, non-monotonic
sch2 = SchurComplementHierarchicalRiskParity(;
                                             params = SchurComplementParams(; gamma = 1,
                                                                            r = r,
                                                                            pdm = nothing,
                                                                            alg = NonMonotonicSchurComplement()))
# Schur complement hierarchical risk parity nearing the mean variance optimisation
sch3 = SchurComplementHierarchicalRiskParity(;
                                             params = SchurComplementParams(; gamma = 1,
                                                                            r = r,
                                                                            alg = MonotonicSchurComplement()))
ress = optimise.([hrp, sch1, mr, sch2, sch3], rd)

5-element Vector{NonFiniteAllocationOptimisationResult}:
 HierarchicalResult
       oe ┼ DataType: DataType
       pr ┼ LowOrderPrior
          │         X ┼ 756×20 Matrix{Float64}
          │        mu ┼ 20-element Vector{Float64}
          │     sigma ┼ 20×20 Matrix{Float64}
          │      chol ┼ nothing
          │         w ┼ nothing
          │       ens ┼ nothing
          │       kld ┼ nothing
          │        ow ┼ nothing
          │        rr ┼ nothing
          │      f_mu ┼ nothing
          │   f_sigma ┼ nothing
          │       f_w ┴ nothing
      clr ┼ Clusters
          │   res ┼ Clustering.Hclust{Float64}([-13 -1; -16 -14; -7 -4; -5 -20; -12 -8; -3 -9; -15 5; 7 -11; -18 1; 9 3; 2 -10; 10 -2; -19 11; 6 4; -6 14; 15 -17; 13 8; 17 12; 18 16], [0.3396697691642121, 0.34762217459549366, 0.4294860942569611, 0.2816354347902738, 0.4214694877772248, 0.17325019149889836, 0.4275113407151469, 0.45669085435759316, 0.4918776056007628, 0.5138138228514841, 0.4134598069341204, 0.558

We can compute the statistics and visualise the results of each estimator.

In [22]:
pr = ress[1].pr
r = factory(Variance(), pr)
rk_rt_ratio = [expected_risk_ret_ratio(r, ArithmeticReturn(), res.w, pr) for res in ress]
rk = map(rr -> rr[1], rk_rt_ratio)
rt = map(rr -> rr[2], rk_rt_ratio)
ratio = map(rr -> rr[3], rk_rt_ratio)
# Display results
pretty_table(hcat(DataFrame(:assets => rd.nx),
                  DataFrame(reduce(hcat, [res.w for res in ress]),
                            ["HRP", "gamma = 0", "MVO", "gamma = 1", "gamma = :max"]));
             formatters = [resfmt])
pretty_table(hcat(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"]),
                  DataFrame(vcat(rk', rt', ratio'),
                            ["HRP", "gamma = 0", "MVO", "gamma = 1", "gamma = :max"]));
             formatters = [resfmt])

┌────────┬─────────┬───────────┬──────────┬───────────┬──────────────┐
│ assets │     HRP │ gamma = 0 │      MVO │ gamma = 1 │ gamma = :max │
│ String │ Float64 │   Float64 │  Float64 │   Float64 │      Float64 │
├────────┼─────────┼───────────┼──────────┼───────────┼──────────────┤
│   AAPL │ 4.612 % │   4.612 % │    0.0 % │   0.258 % │      7.401 % │
│    AMD │ 1.378 % │   1.378 % │    0.0 % │   3.115 % │      0.065 % │
│    BAC │ 3.001 % │   3.001 % │    0.0 % │   0.106 % │      2.364 % │
│    BBY │ 3.707 % │   3.707 % │  0.001 % │     0.0 % │      0.319 % │
│    CVX │ 3.186 % │   3.186 % │    0.0 % │     0.0 % │      0.365 % │
│     GE │ 2.013 % │   2.013 % │    0.0 % │   0.917 % │      0.092 % │
│     HD │ 6.219 % │   6.219 % │    0.0 % │   0.591 % │      9.668 % │
│    JNJ │ 9.316 % │   9.316 % │ 13.325 % │     0.0 % │     14.118 % │
│    JPM │ 3.624 % │   3.624 % │    0.0 % │   0.117 % │      2.943 % │
│     KO │ 5.824 % │   5.824 % │ 20.013 % │     0.0 % │      5.553 % │
│    L

#### 3.3.3 Hierachical equal risk contribution

The [`HierarchicalEqualRiskContribution`]-(@ref) estimator uses the hierarchical structure of the assets as well as a clustering quality score to iteratively break up the asset universe into left and right clusters up until the optimal number of clusters according to the score. Each cluster is treated as a synthetic asset for which their risk is computed. The weight of each cluster is computed based on the risk it represents with respect to the cluster on the other side, and this weight is then multiplied by the weights represented by its member assets which were computed based on the risk they represented in porportion to the other assets in the cluster.

The [`HierarchicalOptimiser`]-(@ref) is specified via the `opt` keyword. Since this optimiser breaks up the assets into intra- and inter-cluster optimisations, it's possible to provide inner and outer risk measures via the `ri` and `ro` keywords, which both default to the `Variance`. The original formulation used equal weights for the inner risk measure.

In [23]:
using Clustering
# Hierarchical equal risk contribution, equal weights risk inner risk measure, variance outer risk measure
herc = HierarchicalEqualRiskContribution(; ri = EqualRiskMeasure(), ro = Variance())
res = optimise(herc, rd)

HierarchicalResult
       oe ┼ DataType: DataType
       pr ┼ LowOrderPrior
          │         X ┼ 756×20 Matrix{Float64}
          │        mu ┼ 20-element Vector{Float64}
          │     sigma ┼ 20×20 Matrix{Float64}
          │      chol ┼ nothing
          │         w ┼ nothing
          │       ens ┼ nothing
          │       kld ┼ nothing
          │        ow ┼ nothing
          │        rr ┼ nothing
          │      f_mu ┼ nothing
          │   f_sigma ┼ nothing
          │       f_w ┴ nothing
      clr ┼ Clusters
          │   res ┼ Clustering.Hclust{Float64}([-13 -1; -16 -14; -7 -4; -5 -20; -12 -8; -3 -9; -15 5; 7 -11; -18 1; 9 3; 2 -10; 10 -2; -19 11; 6 4; -6 14; 15 -17; 13 8; 17 12; 18 16], [0.3396697691642121, 0.34762217459549366, 0.4294860942569611, 0.2816354347902738, 0.4214694877772248, 0.17325019149889836, 0.4275113407151469, 0.45669085435759316, 0.4918776056007628, 0.5138138228514841, 0.4134598069341204, 0.5586690778496974, 0.5383772425351973, 0.47369740075876027, 0.

We can view the results and verify that indeed all assets within a single cluster have the same weight.

In [24]:
pr = res.pr
r = factory(Variance(), pr)
rk, rt, ratio = expected_risk_ret_ratio(r, ArithmeticReturn(), res.w, pr)
# Display results
pretty_table(DataFrame(:assets => rd.nx, :cluster => assignments(res.clr),
                       :Weights => res.w); formatters = [resfmt])
pretty_table(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"],
                       :Measure => [rk, rt, ratio]); formatters = [resfmt])

┌────────┬─────────┬─────────┐
│ assets │ cluster │ Weights │
│ String │   Int64 │ Float64 │
├────────┼─────────┼─────────┤
│   AAPL │       1 │ 2.618 % │
│    AMD │       1 │ 2.618 % │
│    BAC │       2 │ 8.474 % │
│    BBY │       1 │ 2.618 % │
│    CVX │       2 │ 8.474 % │
│     GE │       2 │ 8.474 % │
│     HD │       1 │ 2.618 % │
│    JNJ │       3 │ 4.181 % │
│    JPM │       2 │ 8.474 % │
│     KO │       3 │ 4.181 % │
│    LLY │       3 │ 4.181 % │
│    MRK │       3 │ 4.181 % │
│   MSFT │       1 │ 2.618 % │
│    PEP │       3 │ 4.181 % │
│    PFE │       3 │ 4.181 % │
│      ⋮ │       ⋮ │       ⋮ │
└────────┴─────────┴─────────┘
                5 rows omitted
┌─────────────────┬───────────┐
│            Stat │   Measure │
│          String │   Float64 │
├─────────────────┼───────────┤
│        Variance │   0.027 % │
│          Return │   0.048 % │
│ Return/Variance │ 176.821 % │
└─────────────────┴───────────┘


#### 3.3.4 Nested clustered optimisation

The [`NestedClustered`]-(@ref) optimiser uses the same idea as the [`HierarchicalEqualRiskContribution`]-(@ref), where the optimisation process is split into inner and outer optimisations using the same scoring system for finding the optimal number of clusters. However, unlike [`HierarchicalEqualRiskContribution`]-(@ref), the intra- and inter-cluster optimisations are completely independent. It is possible to provide any non finite allocation optimisation estimator for the inner and outer estimators independently via the keywods `opti` and `opto` respectively. This means it inherits the requirements for the inner and outer estimators respectively.

It is also possible to optimise the outer estimator by using cross validation via the `cv` keyword. A cross validation prediction is applied to the inner estimators, yielding a predicted returns series for each cluster. The returns vector for each cluster is then taken as the returns vector for a synthetic asset, and the resulting returns matrix is used to optimise the outer estimator. Otherwise the returns are computed directly by multiplying the inner weights by the original returns matrix. The final weights are computed in a similar way to the [`HierarchicalEqualRiskContribution`]-(@ref), multiplying weights of each cluster by their corresponding inner weights.

In [25]:
# Emulating the original `HierarchicalEqualRiskContribution`
nco1 = NestedClustered(; opti = EqualWeighted(),
                       opto = RiskBudgeting(; opt = JuMPOptimiser(; slv = slv)))
# Mean risk for both optimisations
nco2 = NestedClustered(; opti = MeanRisk(; opt = JuMPOptimiser(; slv = slv)),
                       opto = MeanRisk(; opt = JuMPOptimiser(; slv = slv)))
# It's even possible to nest them
nco3 = NestedClustered(;
                       opti = NestedClustered(; opti = HierarchicalEqualRiskContribution(;),
                                              opto = RiskBudgeting(;
                                                                   opt = JuMPOptimiser(;
                                                                                       slv = slv))),
                       opto = NestedClustered(;
                                              opti = RiskBudgeting(;
                                                                   opt = JuMPOptimiser(;
                                                                                       slv = slv)),
                                              opto = MeanRisk(;
                                                              opt = JuMPOptimiser(;
                                                                                  slv = slv))))
# Optimise them all in one go
ress = optimise.([nco1, nco2, nco3], rd)

3-element Vector{NestedClusteredResult{DataType, LowOrderPrior{Matrix{Float64}, Vector{Float64}, Matrix{Float64}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing}, Clusters{Clustering.Hclust{Float64}, Matrix{Float64}, Matrix{Float64}, Int64}, WeightBounds{Vector{Float64}, Vector{Float64}}, Nothing, Vector{NonFiniteAllocationOptimisationResult}, T7, Nothing, OptimisationSuccess{Nothing}, Vector{Float64}, Nothing} where T7}:
 NestedClusteredResult
       oe ┼ DataType: DataType
       pr ┼ LowOrderPrior
          │         X ┼ 756×20 Matrix{Float64}
          │        mu ┼ 20-element Vector{Float64}
          │     sigma ┼ 20×20 Matrix{Float64}
          │      chol ┼ nothing
          │         w ┼ nothing
          │       ens ┼ nothing
          │       kld ┼ nothing
          │        ow ┼ nothing
          │        rr ┼ nothing
          │      f_mu ┼ nothing
          │   f_sigma ┼ nothing
          │       f_w ┴ nothing
      clr ┼ Clusters
        

We can compute some risk characteristics and visualise the results. We can see how the analogous optimisation to the original version of [`HierarchicalEqualRiskContribution`]-(@ref) has a similar behaviour, where all assets within a cluster have the same weight as each other.

In [26]:
pr = ress[1].pr
r = factory(Variance(), pr)
rk_rt_ratio = [expected_risk_ret_ratio(r, ArithmeticReturn(), res.w, pr) for res in ress]
rk = map(rr -> rr[1], rk_rt_ratio)
rt = map(rr -> rr[2], rk_rt_ratio)
ratio = map(rr -> rr[3], rk_rt_ratio)
# Display results
pretty_table(hcat(DataFrame(:assets => rd.nx, :clusters => assignments(ress[1].clr)),
                  DataFrame(reduce(hcat, [res.w for res in ress]),
                            ["EW-RB", "MR-MR", "NC-HERC-RB_NC-RB-MR"]));
             formatters = [resfmt])
pretty_table(hcat(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"]),
                  DataFrame(vcat(rk', rt', ratio'),
                            ["EW-RB", "MR-MR", "NC-HERC-RB_NC-RB-MR"]));
             formatters = [resfmt])

┌────────┬──────────┬─────────┬──────────┬─────────────────────┐
│ assets │ clusters │   EW-RB │    MR-MR │ NC-HERC-RB_NC-RB-MR │
│ String │    Int64 │ Float64 │  Float64 │             Float64 │
├────────┼──────────┼─────────┼──────────┼─────────────────────┤
│   AAPL │        1 │ 4.817 % │    0.0 % │             3.008 % │
│    AMD │        1 │ 4.817 % │    0.0 % │             8.832 % │
│    BAC │        2 │  4.26 % │    0.0 % │             2.515 % │
│    BBY │        1 │ 4.817 % │    0.0 % │             3.229 % │
│    CVX │        2 │  4.26 % │    0.0 % │             2.479 % │
│     GE │        2 │  4.26 % │    0.0 % │             5.281 % │
│     HD │        1 │ 4.817 % │    0.0 % │             5.776 % │
│    JNJ │        3 │ 5.692 % │ 13.713 % │             5.509 % │
│    JPM │        2 │  4.26 % │  0.001 % │             3.036 % │
│     KO │        3 │ 5.692 % │ 21.401 % │             4.804 % │
│    LLY │        3 │ 5.692 % │    0.0 % │             7.593 % │
│    MRK │        3 │ 5.6

#### 3.3.5 Stacking optimisation

The [`Stacking`]-(@ref) optimiser uses a similar approach to [`NestedClustered`]-(@ref), but instead of using a single inner estimator, it uses a vector of estimators, inhering the requirements of each estimator being used. The inner weights are optimised by each estimator, and outer estimator uses each inner optimisation as a synthetic asset. The returns series used in the outer optimisation can be computed in the same way as [`NestedClustered`]-(@ref), either directly or by using cross validation predictions. The final weights are computed the same way as well. It is possible to provide a vector of weights for the inner estimators via the `scale` keyword.

[`Stacking`]-(@ref) can be used in [`NestedClustered`]-(@ref) and vice-versa.

The keywords for the inner and outer optimisers are the same as [`NestedClustered`]-(@ref).

In [27]:
# Use a few different optimisers
st = Stacking(;
              opti = [MeanRisk(; opt = JuMPOptimiser(; slv = slv)),
                      RiskBudgeting(; opt = JuMPOptimiser(; slv = slv)),
                      InverseVolatility(), HierarchicalEqualRiskContribution()],
              opto = NearOptimalCentering(; opt = JuMPOptimiser(; slv = slv)))
# Optimise
res = optimise(st, rd)

StackingResult
       oe ┼ DataType: DataType
       pr ┼ LowOrderPrior
          │         X ┼ 756×20 Matrix{Float64}
          │        mu ┼ 20-element Vector{Float64}
          │     sigma ┼ 20×20 Matrix{Float64}
          │      chol ┼ nothing
          │         w ┼ nothing
          │       ens ┼ nothing
          │       kld ┼ nothing
          │        ow ┼ nothing
          │        rr ┼ nothing
          │      f_mu ┼ nothing
          │   f_sigma ┼ nothing
          │       f_w ┴ nothing
       wb ┼ WeightBounds
          │   lb ┼ 20-element Vector{Float64}
          │   ub ┴ 20-element Vector{Float64}
     fees ┼ nothing
     resi ┼ NonFiniteAllocationOptimisationResult[MeanRiskResult
          │        oe ┼ DataType: DataType
          │        pa ┼ ProcessedJuMPOptimiserAttributes
          │           │        pr ┼ LowOrderPrior
          │           │           │         X ┼ 756×20 Matrix{Float64}
          │           │           │        mu ┼ 20-element Vector{Float64

Compute and view the results.

In [28]:
pr = res.pr
r = factory(Variance(), pr)
rk, rt, ratio = expected_risk_ret_ratio(r, ArithmeticReturn(), res.w, pr)
# Display results
pretty_table(DataFrame(:assets => rd.nx, :Weights => res.w); formatters = [resfmt])
pretty_table(DataFrame(:Stat => ["Variance", "Return", "Return/Variance"],
                       :Measure => [rk, rt, ratio]); formatters = [resfmt])

┌────────┬──────────┐
│ assets │  Weights │
│ String │  Float64 │
├────────┼──────────┤
│   AAPL │   0.06 % │
│    AMD │  0.039 % │
│    BAC │  0.069 % │
│    BBY │  0.055 % │
│    CVX │  0.069 % │
│     GE │  0.057 % │
│     HD │  0.068 % │
│    JNJ │ 13.224 % │
│    JPM │  0.078 % │
│     KO │ 19.816 % │
│    LLY │  0.072 % │
│    MRK │ 17.964 % │
│   MSFT │  0.063 % │
│    PEP │  0.082 % │
│    PFE │  6.995 % │
│      ⋮ │        ⋮ │
└────────┴──────────┘
       5 rows omitted
┌─────────────────┬───────────┐
│            Stat │   Measure │
│          String │   Float64 │
├─────────────────┼───────────┤
│        Variance │   0.014 % │
│          Return │   0.057 % │
│ Return/Variance │ 401.593 % │
└─────────────────┴───────────┘


---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*